# Pipeline 3 — VGG16 (ImageNet Pretrained)

**AI vs Real Image Detection** using a fine-tuned VGG16 backbone with dual classification heads.

| Property | Value |
|----------|-------|
| Backbone | VGG16 (ImageNet pretrained) |
| Input size | 224 x 224 |
| Encoder params | 14.7M (shared convolutional features) |
| Trainable params | ~5.5M (blocks 4-5 + heads) |
| Classification | Independent per-face + full image predictions |
| Normalization | ImageNet: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225] |

---
## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Shared notebook utilities
from notebooks import nb_utils

# VGG16 inference pipeline
from inference.vgg16_pipeline import VGG16Pipeline

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch:      {torch.__version__}")
print(f"Device:       {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

---
## 2. Dataset Overview (GRAVEX-200K)

The **GRAVEX-200K** dataset contains 200,000 images:
- **100,000 real** photographs
- **100,000 AI-generated** images

All images include face detection metadata (bounding boxes and pre-cropped face regions) produced by RetinaFace during preprocessing.

In [ ]:
samples = nb_utils.show_dataset_overview()

---
## 3. Preprocessing Pipeline

VGG16 expects **224 x 224** input with **ImageNet normalization**. The preprocessing pipeline applies three steps:

1. **Resize with interpolation** — Lanczos resampling to 224 x 224 preserves high-frequency detail better than bilinear.
2. **Color space analysis** — RGB channels are kept separate; VGG16's early convolutional filters learn channel-specific edge detectors.
3. **Noise reduction** — Optional denoising (Gaussian, median, bilateral) can remove JPEG artifacts without destroying AI-generation fingerprints.

In [ ]:
# Pick a sample image for the preprocessing demo
sample_image_path = samples[0]["image_path"]
print(f"Sample image: {sample_image_path}")

nb_utils.show_preprocessing_steps(sample_image_path, target_size=224)

---
## 4. Augmentation Study

Data augmentation is critical for preventing overfitting and improving generalization. Each augmentation targets a specific invariance:

| Augmentation | Purpose |
|---|---|
| Rotation (15 deg) | Camera tilt invariance — keeps angle small to preserve face geometry |
| Horizontal Flip | Doubles effective dataset — vertical flip omitted (breaks gravity cues) |
| Random Crop | Simulates different framing and zoom levels |
| Translation | Position invariance — shifts image up to 10% in each axis |
| Gaussian Blur | Defocus/motion blur robustness — model cannot rely on sharpness alone |
| Sharpening | Edge enhancement — over-sharpening halos can mimic AI artifacts |
| Color Jitter | Lighting/white balance variation — prevents learning brightness as a proxy |
| JPEG Compression | Social media re-encoding robustness — prevents learning compression artifacts |

In [ ]:
nb_utils.show_augmentation_study(sample_image_path, size=224)

---
## 5. Model Architecture — VGG16

**VGG16** is a classic deep CNN with a simple, uniform architecture: 13 convolutional layers organized into 5 blocks, followed by 3 fully connected layers (which we replace with our custom dual heads).

### Architecture

```
Input (3 x 224 x 224)
    |
    v
[Block 1] Conv3-64 x2 + MaxPool   -> 64 x 112 x 112   (FROZEN - edges, gradients)
[Block 2] Conv3-128 x2 + MaxPool  -> 128 x 56 x 56    (FROZEN - textures)
[Block 3] Conv3-256 x3 + MaxPool  -> 256 x 28 x 28    (FROZEN - patterns)
[Block 4] Conv3-512 x3 + MaxPool  -> 512 x 14 x 14    (TRAINABLE - parts)
[Block 5] Conv3-512 x3 + MaxPool  -> 512 x 7 x 7      (TRAINABLE - objects)
    |
    v
AdaptiveAvgPool -> 512-dim feature vector
    |
    +-----> face_head: Linear(512->256) -> ReLU -> Dropout(0.5) -> Linear(256->1)
    |
    +-----> full_head: Linear(512->256) -> ReLU -> Dropout(0.5) -> Linear(256->1)
```

### Freezing Strategy

- **Blocks 1-3 are frozen** (edges, textures, patterns) — these learn universal ImageNet features that transfer perfectly to any image domain.
- **Blocks 4-5 are trainable** — higher-level features adapt to detect AI-specific artifacts.
- **Both classification heads are trainable** — separate weights for face crops vs full images.

### Parameter Budget

- Encoder (features): **14.7M** total params
- Trainable: **~5.5M** (blocks 4-5 + both heads)
- Frozen: **~9.5M** (blocks 1-3)

In [ ]:
from vgg16_detector.vgg16_dual_branch import VGG16DualBranchDetector

model = VGG16DualBranchDetector(pretrained=True, freeze_blocks=3)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters:    {frozen:,} ({frozen/total:.1%})")
print(f"\nEncoder:   {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"Face head: {sum(p.numel() for p in model.face_head.parameters()):,}")
print(f"Full head: {sum(p.numel() for p in model.full_head.parameters()):,}")

---
## 6. Training Configuration

| Hyperparameter | Value |
|---|---|
| Epochs | 30 |
| Batch size | 256 (GPU) |
| Learning rate (heads) | 1e-4 |
| Learning rate (encoder) | 1e-5 (differential lr: encoder x 0.1) |
| Optimizer | AdamW |
| Scheduler | Cosine Annealing + Warmup |
| Loss | BCEWithLogitsLoss |
| Precision | 16-mixed (AMP) |
| Early stopping patience | 7 |
| Frozen blocks | 1-3 (edges, textures, patterns) |

**Differential learning rate**: The encoder backbone uses a 10x lower learning rate than the classification heads. This prevents catastrophic forgetting of pretrained ImageNet features while allowing the heads to learn quickly.

In [ ]:
import pandas as pd

metrics_path = PROJECT_ROOT / "results" / "vgg16" / "val_metrics.csv"

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curve
    if "val_loss" in df.columns:
        axes[0].plot(df["epoch"], df["val_loss"], "b-o", markersize=3, label="Val Loss")
        if "train_loss" in df.columns:
            axes[0].plot(df["epoch"], df["train_loss"], "r--", alpha=0.6, label="Train Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training & Validation Loss")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

    # Accuracy curve
    if "val_acc" in df.columns:
        axes[1].plot(df["epoch"], df["val_acc"], "g-o", markersize=3, label="Val Accuracy")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].set_title("Validation Accuracy")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    # AUC curve
    if "val_auc" in df.columns:
        axes[2].plot(df["epoch"], df["val_auc"], "m-o", markersize=3, label="Val AUC-ROC")
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylabel("AUC-ROC")
        axes[2].set_title("Validation AUC-ROC")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

    plt.suptitle("VGG16 Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"Best validation loss:     {df['val_loss'].min():.4f} (epoch {df['val_loss'].idxmin()})")
    if "val_acc" in df.columns:
        print(f"Best validation accuracy: {df['val_acc'].max():.4f} (epoch {df['val_acc'].idxmax()})")
    if "val_auc" in df.columns:
        print(f"Best validation AUC:      {df['val_auc'].max():.4f} (epoch {df['val_auc'].idxmax()})")
else:
    print("Training metrics not available yet — VGG16 is still training on RunPod.")
    print(f"Expected path: {metrics_path}")

---
## 7. Sample Predictions

In [ ]:
pipeline = VGG16Pipeline()

if pipeline.is_available():
    print(f"VGG16 pipeline loaded successfully.")
    print(f"Showing predictions on 8 random test samples...\n")
    nb_utils.show_predictions(pipeline, samples, n=8)
else:
    print("VGG16 pipeline is NOT available — training is still in progress on RunPod.")
    print("Once training completes, export weights to: weights/vgg16_finetuned.pt")
    print("Then re-run this cell to see predictions.")

---
## 8. Evaluation Metrics

In [ ]:
if pipeline.is_available():
    # Check for cached results first
    cached_path = PROJECT_ROOT / "results" / "vgg16" / "eval_cache.npz"

    if cached_path.exists():
        print("Loading cached evaluation results...")
        data = np.load(cached_path)
        y_true, y_scores = data["y_true"], data["y_scores"]
    else:
        print("Running evaluation on test set (this may take a few minutes)...")
        y_true, y_scores = nb_utils.evaluate_pipeline(pipeline, samples, max_images=2000)
        # Cache results
        cached_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cached_path, y_true=y_true, y_scores=y_scores)
        print(f"Results cached to {cached_path}")

    vgg16_results = nb_utils.show_metrics(y_true, y_scores, model_name="VGG16")
else:
    print("VGG16 is still training on RunPod — evaluation metrics are not available yet.")
    print()
    print("Expected metrics after training:")
    print("  - Confusion matrix (Real vs AI classification)")
    print("  - ROC curve with AUC score")
    print("  - Accuracy, Precision, Recall, F1")
    print()
    print("Placeholder — results will be populated once weights/vgg16_finetuned.pt is available.")

---
## 9. Inference Speed

In [ ]:
if pipeline.is_available():
    avg_time = nb_utils.show_inference_speed(pipeline, samples, n_runs=50)
else:
    print("VGG16 pipeline not available — skipping inference speed benchmark.")
    print("Re-run this cell after training completes and weights are exported.")

---
## 10. Summary

**VGG16** uses ImageNet pretrained convolutional features as a shared encoder for AI vs Real image detection.

### Key Design Decisions

- **Blocks 1-3 are frozen** (edges, textures, patterns) — these learn universal visual features that transfer perfectly across domains. Freezing them saves ~60% of backward computation and activation memory, enabling larger batch sizes.
- **Only blocks 4-5 and both classification heads are fine-tuned** (~5.5M trainable parameters out of ~15M total). The higher-level convolutional features in blocks 4-5 adapt to detect AI-specific generation artifacts.
- **Differential learning rate** (encoder x 0.1, heads x 1.0) prevents catastrophic forgetting of pretrained features while allowing rapid head convergence.
- **Dual-head architecture** with independent predictions — each detected face gets its own AI/Real classification with a confidence score, and the full image gets a separate prediction. No fusion is applied.

### Comparison with Other Pipelines

VGG16 serves as a **baseline CNN approach** compared to the transformer-based CLIP pipeline. While VGG16 has fewer parameters, its purely convolutional architecture may miss global context that Vision Transformers capture through self-attention. Convolutions have a limited receptive field that grows slowly with depth, whereas ViT attention is global from the first layer.

Results will be compared with CLIP and FerretNet once training completes on RunPod.